# N012 · 从约束推导可行算法

**目标：** 把约束当作筛选算法的依据而非死记规模表。

**先修：** N011。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 输入维度；值域；稀疏性；指数状态；常数与运行环境；输出规模。

**配套讲解来源：** [同名逐章意见](../../comment/012_constraints_to_algorithms.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

刷题时你一定见过题目末尾的"数据范围"（比如 n ≤ 20 或 n ≤ 10⁵）。很多人背一张"n 多大用多少复杂度"的表格去应试，这一章要解决的问题是：**别死背表格，学会把约束当作"筛选算法的线索"，自己动手把操作总数算出来，再判断某个算法到底来不来得及。**

核心观念是三段论（“思路推导”部分 原话翻译过来）：

1. 先**精确数出**组合的数量（比如子集有几个、数对有几对）；
2. 再**乘上每个组合要做的**工作量（每个状态可能还要再做 O(n) 的活，不是数完就完了）；
3. 最后才**对照机器预算**（一秒大约能做多少次基本操作）下结论。

给你一个具体到数字的例子：题目说 n = 20，能不能用"枚举所有子集"的暴力法？

- 我们先数状态数：每个元素要么取要么不取，20 个元素共有 2²⁰ = 1,048,576 个子集，约一百万个状态。
- 光看"一百万"好像可行？慢着，还要乘每状态的工作量：如果每生成一个子集还要花 O(n)=20 步去检查它，总操作数就是约 2000 万次——一般机器一两秒内能扛住。但如果每状态要做的是 O(2ⁿ) 或更重的活，n=20 立刻爆炸。
- 所以结论是"看情况"，而看情况的依据就是把账算出来。**约束是线索，不是保票。**

同样的思路反过来用一次：约束还能帮我们**排除**算法。还是上面的例子，如果题目写明 n ≤ 40，那么任何 2ⁿ 状态的枚举（2⁴⁰ 约一万亿）当场出局，不管常数多小都救不回来；如果 n ≤ 20，2ⁿ 枚举至少进了候选名单，但要再过"乘每状态代价"这一关。也就是说，约束是一张双向筛子：一边放算法进候选，一边把明显不可行的筛掉。

## 2. 基础知识：先读懂这些词

- **输入维度（input dimensions）**：题目里有几个会变化的规模量——数组长度 n、数对个数 m、查询次数 q。判断复杂度前先问"到底有几个维度"，因为 O(nm) 和 O(n²) 是完全不同的两回事。
- **值域（value range）**：数值本身的大小范围，比如 0 ≤ nums[i] ≤ 100。值域小意味着可以开一个按值索引的数组（计数数组、桶），这是"用值域换时间"的思路来源。
- **稀疏性（sparsity）**：实际数据量远小于理论上限。比如 n=10⁵ 个节点的图，边数可能只有几千条——这时复杂度里的主导参数可能是边数 m 而不是 n²，算法选择也随之改变。
- **指数状态（exponential states）**：子集枚举这类随 n 翻倍、状态数翻平方的状态空间，n 个元素的子集共 2ⁿ 个。n 每加 1，状态数翻一倍，所以 n=20 和 n=40 之间隔了一百万倍的鸿沟。
- **常数与运行环境（constants and environment）**：同样 O(n) 的两个算法，一次做加法、一次做字符串拼接，实际速度可能差十倍；在线评测机的每秒操作数也和你笔记本不同。所以算出的操作数只是模型，"这个规模一定能过"是不做承诺的（“正确性与复杂度”部分 原话）。
- **输出规模（output size）**：如果题目要求输出本身就是 O(n²) 大小（比如所有数对），那任何算法都不可能快过 O(n²)——输出写得慢，算法就慢，复杂度下限由输出决定。
- **无序对（pairs）**：从 n 个位置里挑两个不分次序的组合数 n(n−1)/2，上一章（N011）推导过的公式，本章直接复用。
- **左移运算 `1 << n`**：把二进制的 1 左移 n 位，结果就是 2ⁿ。用位运算算 2ⁿ 既快又直观（n 个比特每个两种取值，正好对应子集）。

## 3. 思路推导：从小例子开始

本章的"算法"其实是一套**评估流程**，代码是把这套流程落到数字上：

- **Step 1：读约束，找出所有维度。** 问自己：几个元素？值多大？数据是稠密还是稀疏？有没有查询次数这类第二维度？
- **Step 2：为候选算法数状态数。** 线性扫描是 n 个状态；枚举数对是 n(n−1)/2 个；枚举子集是 2ⁿ 个。本章 `count_naive_operations` 一并返回这三个数。
- **Step 3：乘上每状态工作量。** 状态数 × 单状态代价 = 总操作数。“思路推导”部分 特意提醒：n=20 的子集枚举约百万个状态，但每状态还可能做 O(n) 的工作，不能只看状态数沾沾自喜。
- **Step 4：对照预算下结论。** 经验上，评测机一秒约能做 10⁷ 到 10⁸ 次基本操作。总操作数在 10⁷ 以内基本稳，超过 10⁹ 就危险。同时记得 “思路推导”部分 的另一句：n=10⁵ 的大输入也可能因为数据稀疏而改用依赖别的参数的算法——先看实际数据的形状再定。

拿 n=10 把整套流程空跑一遍，你要能在脑子里报出这三个数：linear=10、pairs=45、subsets=1024。报得出来，说明公式已经长在你手上了；接着乘每状态代价（比如每状态 O(n)=10），子集枚举总数约 10⁴——预算内轻松可行。

**手算演示（n = 4，自己数一遍三个量）：**

- 线性扫描：4 个元素各看一次，4 步。
- 数对：(0,1)(0,2)(0,3)(1,2)(1,3)(2,3) 共 6 对，公式 4×3/2=6。
- 子集：{}、{0}、{1}、{2}、{3}、{0,1}、{0,2}、{0,3}、{1,2}、{1,3}、{2,3}、{0,1,2}、{0,1,3}、{0,2,3}、{1,2,3}、{0,1,2,3}，共 16 个，公式 2⁴=16。

**“运行示例”部分 表格数据（增长对比，n 取 4 到 20）：**

| n | 线性 | 索引对 | 子集 |
|---|------|--------|------|
| 4 | 4 | 6 | 16 |
| 8 | 8 | 28 | 256 |
| 12 | 12 | 66 | 4,096 |
| 16 | 16 | 120 | 65,536 |
| 20 | 20 | 190 | 1,048,576 |

盯住"子集"那一列：n 从 16 到 20 只增加了 4，状态数却从约 6.5 万暴涨到约 105 万。线性那一列则是老老实实一步一格地涨。这就是"指数增长"和"线性增长"同框的画面。

再把 Step 3"乘每状态工作量"落成一张账单（固定 n=20，只变单状态代价）：

| 每个状态要做的事 | 单状态代价 | 总操作数 | 一秒预算（约 10⁸）下 |
|------------------|------------|----------|----------------------|
| 只数一下 | O(1) | 约 1.05×10⁶ | 稳过 |
| 扫一遍子集求和 | O(n)=20 | 约 2.1×10⁷ | 大概率可行 |
| 对子集做 O(n²) 检查 | 400 | 约 4.2×10⁸ | 危险 |
| 每状态再枚举一层子集 | O(2ⁿ) | 约 10¹² | 完全不可行 |

同一句"n=20 能不能暴力"，四种答案——差别全在你给每个状态派了什么活。这就是"先数组合数、再乘每状态工作量"这套流程存在的意义。

## 4. 核心代码：count_naive_operations

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def count_naive_operations(n):
    if n < 0:
        raise ValueError('n must be nonnegative')
    return {'linear': n, 'pairs': n * (n - 1) // 2, 'subsets': 1 << n}
```

### 逐段读懂代码

“分段实现”部分 定义了两个函数。

### 函数 1：`count_naive_operations` —— 一次算出三个规模量

```python
def count_naive_operations(n):
    if n < 0:
        raise ValueError('n must be nonnegative')
    return {'linear': n, 'pairs': n * (n - 1) // 2, 'subsets': 1 << n}
```

- `if n<0: raise ValueError('n must be nonnegative')`：这是输入合法性检查，专门挡负数——规模不可能是负的，传负数进来属于调用方的错误，直接抛异常把问题暴露出来，而不是让它悄悄算出怪结果（比如 1<<(-1) 会直接崩）。
- `return {'linear':n, ...}`：一次性返回一个字典，装三个关键量：linear 是线性扫描的操作数 n；pairs 是无序对数 n(n−1)/2（整数除法 `//`，n(n−1) 必为偶数所以无损）；subsets 用 `1<<n` 即 2ⁿ 算子集数。
- 为什么用字典而不是三个返回值？因为调用方常常只要其中一个量（比如 `d['subsets']`），字典按键取名最顺手，“运行示例”部分 和 “自动测试”部分 都是这么用的。
- 为什么要 `raise` 而不是悄悄返回 0？因为"规模是负数"说明调用方代码已经写错了，这时返回一个看似正常的数字会把错误掩盖到更远的地方才爆，越晚越难查——早爆早修，这是 N010 调试思想的延续。
- 顺带说 `1<<n` 这个写法：左移一位相当于二进制末尾添个 0、数值翻倍，所以 1<<n 恰好等于 2ⁿ；它和"每个元素取/不取两种选择"的二叉决策树严丝合缝。

### 函数 2：`compare_growth` —— 批量生成对比表

```python
def compare_growth(ns):
    return [(n, count_naive_operations(n)) for n in ns]
```

- 这是个列表推导式：对传入的每个规模 n（比如 [4,8,12,16,20]），调用一次 `count_naive_operations(n)`，把 (n, 字典) 配对收集成一个列表。
- 它的用途就是把第三节的对比表造出来——“运行示例”部分 拿它的输出拆出 linear/pairs/subsets 三列喂给 `show_table`。
- 算术代价本身极小：k 个规模就算 k 次公式，所以 “正确性与复杂度”部分 说表格含 k 个输入规模时需要 O(k) 条记录。

## 5. 分段实现：按顺序运行

**本章接口：** `count_naive_operations(n)`、`compare_growth(ns)`

### count_naive_operations

In [1]:
def count_naive_operations(n):
    if n < 0:
        raise ValueError('n must be nonnegative')
    return {'linear': n, 'pairs': n * (n - 1) // 2, 'subsets': 1 << n}

### compare_growth

In [2]:
def compare_growth(ns):
    return [(n, count_naive_operations(n)) for n in ns]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['n','线性','索引对','子集'],[(n,d['linear'],d['pairs'],d['subsets']) for n,d in compare_growth([4,8,12,16,20])])

n,线性,索引对,子集
4,4,6,16
8,8,28,256
12,12,66,4096
16,16,120,65536
20,20,190,1048576


## 7. 正确性、适用条件与复杂度

每个索引独立选择取或不取，共 2ⁿ 种选择；无序对数来自上章求和。这里是操作模型，不承诺某规模一定能通过在线评测。

**代价：** 表格含 k 个输入规模时需要 O(k) 条记录；精确大整数表示还依赖数值位数。

### 展开理解

**为什么子集数是 2ⁿ？** 想象 n 个元素排成一排，每个元素独立做决定："进子集"或"不进子集"，两种选择互不影响。第 1 个元素 2 种选法，第 2 个元素不管前面怎么选它仍有 2 种……n 个元素的选择组合相乘就是 2×2×…×2 = 2ⁿ。每个子集对应唯一的一串选择，所以不多不少正好 2ⁿ 个。

**为什么无序对数是 n(n−1)/2？** 这是上一章的结论直接搬来：双循环枚举 i<j，内层长度 n−1、n−2、…、1、0，加起来就是 n(n−1)/2。

**这套估算的边界在哪？** “正确性与复杂度”部分 特意说了两层：第一，这是一个"操作模型"，算的是操作次数，不是运行时间——它不承诺"n 等于某个数就一定能通过在线评测"，因为评测机速度、语言、常数因子都会影响真实耗时；第二，当 n 大到 2ⁿ 超出机器字长（比如 n=100），Python 会自动切换成大整数表示，此时连"存下这个数"本身的代价都随位数增长，数字要按位数看。

**复杂度是多少？** 这两个函数本身不是被评估的算法，而是评估工具，它们自己的代价极低：`count_naive_operations` 是 O(1) 的几次算术（把大整数位数考虑进来则是 O(n/64) 级别的位操作）；`compare_growth` 对 k 个规模各调一次，是 O(k) 条记录。

**别忘了一个容易被忽略的下界：输出规模。** 如果题目要求你把答案本身全部打印/返回出来（比如"列出所有数对"），那么哪怕算法聪明到零计算，光写输出就要 n(n−1)/2 份。这种时候约束里 n 的大小其实是在限制"输出有多大"，再快的算法也快不过"把结果写出来"这一步。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的每条断言：

- `assert count_naive_operations(8)['subsets']==256`：**正常值测试**。2⁸=256，验证左移算 2ⁿ 没算错位数。
- `assert count_naive_operations(10)['pairs']==45`：**正常值测试**。10×9/2=45，验证对数公式和整数除法都正确。
- `assert count_naive_operations(0)=={'linear':0,'pairs':0,'subsets':1}`：**边界值测试（n=0）**。零个元素：没有扫描操作（linear=0），凑不出对（pairs=0），但"空集"本身是一个合法子集（subsets=1）。这条最容易凭直觉写错——空输入不是 0 个子集而是 1 个。
- `for n in range(8): assert count_naive_operations(n)['subsets']==len(range(1<<n))`：**交叉验证测试**。对 n=0 到 7 逐个核对：公式 `1<<n` 是否等于真实构造 `range(1<<n)` 后数出来的长度。和 N011 一样，公式必须和实数对上才算数。
- 最后的 `print('N012: 所有本章断言通过')` 不是测试，只是把"全部通过"打在屏幕上，方便你肉眼确认这批断言确实整体跑过了。

四条断言按类型归档：

| 断言 | 类型 | 它防的是哪类 bug |
|------|------|------------------|
| `8 → subsets 256` | 正常值 | 左移位数算错 |
| `10 → pairs 45` | 正常值 | 对数公式/整除写错 |
| `0 → {0,0,1}` | 边界值 | 把空集数成 0 |
| n=0..7 公式 vs 实数 | 交叉验证 | 公式与真实枚举不符 |

In [4]:
assert count_naive_operations(8)['subsets'] == 256

assert count_naive_operations(10)['pairs'] == 45

assert count_naive_operations(0) == {'linear': 0, 'pairs': 0, 'subsets': 1}

for n in range(8):
    assert count_naive_operations(n)['subsets'] == len(range(1 << n))

print('N012: 所有本章断言通过')

N012: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 相同n下比较稀疏图与稠密图。

**练习 2：** 解释n=20不保证所有指数算法可行。

<details>
<summary>展开思路提示（不是完整答案）</summary>

“练习”部分 的两道练习：

- **练习 1（相同 n 下比较稀疏图与稠密图）**：固定 n（比如 5 个节点），分别手画一张稀疏图（比如只有 3 条边）和一张稠密图（比如 9 条边）。提示：遍历相邻结构的算法，工作量正比于"节点数 + 边数"而不只是节点数；稠密时边数接近 n(n−1)/2，稀疏时可能只有 O(n)。先手数两种图各要摸多少条边，再对比同样 n 下两份账单的差别。边界情况：0 条边的图。写结论时记得点题：这正是 “思路推导”部分 说的"n=10⁵ 也可能因输入稀疏而改用依赖其他参数的算法"。
- **练习 2（解释 n=20 不保证所有指数算法可行）**：拿本章表格当证据。提示分两步：第一步，算出 2²⁰ ≈ 1.05×10⁶ 个状态，这个数本身在一秒预算内；第二步，给每个状态乘上不同的单状态代价——乘 O(n)=20，总数约 2×10⁷，可行；乘 O(n·2ⁿ) 或每次状态要做一次 O(n²) 检查，总数立刻上天。结论要落在"可行性 = 状态数 × 单状态代价，缺一不可"这句话上，并配一个手算数字示例。可以直接引用第三节的 n=20 账单表当论据。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def count_naive_operations(n):
    if n < 0:
        raise ValueError('n must be nonnegative')
    return {'linear': n, 'pairs': n * (n - 1) // 2, 'subsets': 1 << n}

def compare_growth(ns):
    return [(n, count_naive_operations(n)) for n in ns]

# 示例与回归测试
assert count_naive_operations(8)['subsets'] == 256

assert count_naive_operations(10)['pairs'] == 45

assert count_naive_operations(0) == {'linear': 0, 'pairs': 0, 'subsets': 1}

for n in range(8):
    assert count_naive_operations(n)['subsets'] == len(range(1 << n))

print('N012: 所有本章断言通过')

N012: 所有本章断言通过


## 11. 代表题与迁移

“题目映射”部分 列出的题目：

- **217. Contains Duplicate（存在重复元素，comparison）**：练的是"用约束选工具"——值域/数据量提示你该用集合（O(n) 时间 O(n) 空间）还是先排序再比相邻（O(n log n) 时间 O(1) 额外空间），两种解法的取舍正是本章"约束当线索"的演示。
- **215. Kth Largest Element in an Array（数组中第 K 个最大，preview）**：预告题——同样的问题，快速选择平均 O(n)、堆 O(n log k)、排序 O(n log n)，选哪个取决于 n、k 的约束关系，等你学到对应章节再回头体会。
- 做这两题时不妨回头补一步：把每种解法的总操作数用本章流程算成具体数字再比较，练"算账"而不是"背表"。

照 notebook 原文提醒：以上条目用于知识映射，本章实现遵循教学契约，刷题时以 LeetCode 官方题面与评测为准。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。